In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re

import nltk
from nltk.corpus import stopwords
# nltk.download('stopwords')
# nltk.download('punkt')

STOPWORDS = set(stopwords.words('english'))

## Removing stopwords and saving adjectives

In [ ]:
def save_adjectives(filename, dep_type, inferred):
    if inferred:
        inferred_type = "_inferred"
    else: 
        inferred_type = ""

    df= pd.read_json(filename, lines=True)

    df_no_stopwords = df[~df["word"].str.lower().isin(STOPWORDS)].sort_values(by="total_count", ascending=False)

    df_adjs = df_no_stopwords[df_no_stopwords["majority_upos"] == "ADJ"]
    
    df_adjs.to_json(
        f"adjectives/adjectives_{dep_type}{inferred_type}.ndjson",
        orient="records",
        lines=True
    )
    
    


### not inferred (includes chatgpt as model)

In [ ]:
save_adjectives("deps/big_corpus/direct/big_corpus_direct_cleaned_deps_mostfreqpos.ndjson", "direct", False)

save_adjectives("deps/big_corpus/onehop/big_corpus_onehop_cleaned_deps_mostfreqpos.ndjson", "onehop", False)

### inferred (chatgpt entries distributed among other models based on model_inferred_temporal)

In [ ]:
save_adjectives("deps/big_corpus/direct/big_corpus_direct_cleaned_deps_mostfreqpos_inferred.ndjson", "direct", True)

save_adjectives("deps/big_corpus/onehop/big_corpus_onehop_cleaned_deps_mostfreqpos_inferred.ndjson", "onehop", True)


## Saving adjectives by model

In [ ]:
def save_adjectives_by_model(filename, dep_type, inferred):
    
    if inferred:
        inferred_type = "inferred"
    else: 
        inferred_type = "not_inferred"
    

    df= pd.read_json(filename, lines=True)

    df_no_stopwords = df[~df["word"].str.lower().isin(STOPWORDS)].sort_values(by="total_count", ascending=False)

    df_adjs = df_no_stopwords[df_no_stopwords["majority_upos"] == "ADJ"]

    unique_models = df_adjs["majority_model"].unique()
    for model in unique_models:
        df_model = df_adjs[df_adjs["majority_model"] == model].sort_values(by="total_count", ascending=False)

        model_type = model.replace("-", "_").replace(".", "_")

        df_model.to_json(f"adjectives/by_model/{dep_type}/{inferred_type}/{model_type}_adjectives_{dep_type}{inferred_type}.ndjson", orient="records", indent=4, lines=True) 

In [ ]:
save_adjectives_by_model("deps/big_corpus/direct/big_corpus_direct_cleaned_deps_mostfreqpos.ndjson", "direct")

In [75]:
import os

# 1. Define the "GPT-4 Family" (EXCLUDING GPT-4O)
gpt4_family = ["gpt-4.5", "gpt-4.1", "gpt-4-turbo", "gpt-4"]

# 2. Save the Consolidated GPT-4 Group
df_gpt4_group = df_direct_adjectives[df_direct_adjectives["majority_model"].isin(gpt4_family)]
df_gpt4_group = df_gpt4_group.sort_values(by="total_count", ascending=False)
df_gpt4_group.to_json(f"adjectives/by_model/direct/inferred/gpt4_family_consolidated_direct_inferred.json", orient="records", indent=4)

# 3. Save everything else individually (including GPT-4o)
unique_models = df_direct_adjectives["majority_model"].unique()

for model in unique_models:
    # Skip the ones we already handled in the consolidated group
    if model in gpt4_family:
        continue
        
    # Filter for this specific model (e.g., GPT-4O, GPT-5, GPT-3.5)
    df_temp = df_direct_adjectives[df_direct_adjectives["majority_model"] == model]
    
    # Sort by frequency so the most common adjectives are at the top
    df_temp = df_temp.sort_values(by="total_count", ascending=False)
    
    # Create a safe filename (e.g., "gpt-4o" -> "gpt_4o_adjectives.json")
    safe_name = model.replace("-", "_").replace(".", "_")
    df_temp.to_json(f"adjectives/by_model/direct/inferred/{safe_name}_adjectives_direct_inferred.json", orient="records", indent=4)



In [3]:
df_mostfreq_direct = pd.read_json("deps/big_corpus/direct/big_corpus_direct_cleaned_deps_mostfreqpos.ndjson", lines=True)

df_mostfreq_onehop = pd.read_json("deps/big_corpus/onehop/big_corpus_onehop_cleaned_deps_mostfreqpos.ndjson", lines=True)

In [4]:
df_direct_wostopwords = df_mostfreq_direct[~df_mostfreq_direct["word"].str.lower().isin(STOPWORDS)].sort_values(by="total_count", ascending=False)

df_onehop_wostopwords = df_mostfreq_onehop[~df_mostfreq_onehop["word"].str.lower().isin(STOPWORDS)].sort_values(by="total_count", ascending=False)

In [7]:
df_direct_adjectives = df_direct_wostopwords[df_direct_wostopwords["majority_upos"] == "ADJ"]
#df_direct_adjectives.to_json("direct_adjectives.json", orient="records", indent=4)

df_onehop_adjectives = df_onehop_wostopwords[df_onehop_wostopwords["majority_upos"] == "ADJ"]
#df_onehop_adjectives.to_json("onehop_adjectives.json", orient="records", indent=4)

df_temp = df_onehop_adjectives[df_onehop_adjectives["majority_model"] == 'gpt-3.5']
df_temp

,word,total_count,majority_upos,majority_count,majority_model,majority_model_count,model_freqs


In [70]:
import os

# 1. Define the "GPT-4 Family" (EXCLUDING GPT-4O)
gpt4_family = ["gpt-4.5", "gpt-4.1", "gpt-4-turbo", "gpt-4"]

# 2. Save the Consolidated GPT-4 Group
df_gpt4_group = df_onehop_adjectives[df_onehop_adjectives["majority_model"].isin(gpt4_family)]
df_gpt4_group = df_gpt4_group.sort_values(by="total_count", ascending=False)
df_gpt4_group.to_json(f"adjectives/by_model/onehop/not_inferred/gpt4_family_consolidated_onehop.json", orient="records", indent=4)

# 3. Save everything else individually (including GPT-4o)
unique_models = df_onehop_adjectives["majority_model"].unique()

for model in unique_models:
    # Skip the ones we already handled in the consolidated group
    if model in gpt4_family:
        continue
        
    # Filter for this specific model (e.g., GPT-4O, GPT-5, GPT-3.5)
    df_temp = df_onehop_adjectives[df_onehop_adjectives["majority_model"] == model]
    
    # Sort by frequency so the most common adjectives are at the top
    df_temp = df_temp.sort_values(by="total_count", ascending=False)
    
    # Create a safe filename (e.g., "gpt-4o" -> "gpt_4o_adjectives.json")
    safe_name = model.replace("-", "_").replace(".", "_")
    df_temp.to_json(f"adjectives/by_model/onehop/not_inferred/{safe_name}_adjectives_onehop.json", orient="records", indent=4)